# Querying Gaia DR3 around the Pleiades

This notebook retrieves and validates a broad Gaia DR3 sample around the
Pleiades, saves the unmodified catalogue results, and produces initial
parallax and proper-motion visualizations.

No narrow parallax or proper-motion membership cuts are applied. The raw
sample retains the surrounding field population needed for later cleaning
and clustering.

## Environment and imports

The interpreter path is printed so the active notebook kernel can be
checked directly. `Path` handles project paths; Astropy represents
coordinates and tables; PyVO accesses a standard TAP service; NumPy and
Matplotlib support validation and plotting.

In [ ]:
import sys
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import pyvo
from astropy.coordinates import SkyCoord
from astropy.table import Table, vstack

print(sys.executable)
print(sys.version)

In [ ]:
project_root = Path.cwd()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"
figures_dir = project_root / "figures"

raw_data_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")

## Search region and query scope

The approximate cluster centre is represented in the ICRS frame. An
initial 5-degree count returned 718,445 sources and proved impractical to
download reliably. A 3-degree radius returned 255,553 sources and remains
broad enough to include both the cluster and a substantial field
population.

Changing the radius was a computational-scope decision, not an astrometric
membership cut.

In [ ]:
pleiades_centre = SkyCoord(
    ra=56.75 * u.deg,
    dec=24.12 * u.deg,
    frame="icrs",
)
search_radius = 3.0 * u.deg

print(pleiades_centre)
print(f"Adopted search radius: {search_radius}")

## Gaia DR3 columns

The query includes sky position, the three astrometric features intended
for later clustering, their uncertainties, Gaia photometry for the future
colour-magnitude diagram, and several quality diagnostics.

Quality columns are retrieved but not filtered here. Missing values and
quality criteria belong in the cleaning notebook.

In [ ]:
gaia_columns = [
    "source_id",
    "ra",
    "dec",
    "parallax",
    "parallax_error",
    "pmra",
    "pmra_error",
    "pmdec",
    "pmdec_error",
    "phot_g_mean_mag",
    "phot_bp_mean_mag",
    "phot_rp_mean_mag",
    "ruwe",
    "visibility_periods_used",
    "astrometric_params_solved",
    "phot_bp_rp_excess_factor",
]

print(f"Selected columns: {len(gaia_columns)}")

## TAP service and ADQL construction

Repeated large requests to the ESA Gaia Archive stalled while that service
was unstable. The remaining downloads were completed through ARI
Heidelberg's Gaia TAP mirror. Both services expose the same
`gaiadr3.gaia_source` catalogue; the endpoint changes, not the Gaia data
release.

The cached workflow below uses ARI only when a required raw chunk is
missing. Existing validated FITS files are loaded locally, making notebook
reruns safe and avoiding duplicate remote jobs.

In [ ]:
ari_gaia = pyvo.dal.TAPService(
    "https://gaia.ari.uni-heidelberg.de/tap"
)

def build_chunk_query(chunk_condition):
    # Build the Gaia DR3 ADQL query for one spatial chunk.

    select_clause = ",\n    ".join(gaia_columns)

    return f''' 
    SELECT
        {select_clause}
    FROM
        gaiadr3.gaia_source
    WHERE
        1 = CONTAINS(
            POINT('ICRS', ra, dec),
            CIRCLE(
                'ICRS',
                {pleiades_centre.ra.deg},
                {pleiades_centre.dec.deg},
                {search_radius.to_value(u.deg)}
            )
        )
        AND {chunk_condition}
    '''

## Non-overlapping download chunks

The 3-degree circle is divided into five non-overlapping pieces to keep
individual archive jobs manageable. The northeast quadrant is split once
more at RA = 58.25 degrees. The use of `<` on one side of each boundary and
`>=` on the other assigns every source to exactly one chunk.

These boundaries alter only the transfer strategy. Every query retains the
same 3-degree circular spatial condition and applies no astrometric cuts.

In [ ]:
chunk_conditions = {
    "south_west": "ra < 56.75 AND dec < 24.12",
    "north_west": "ra < 56.75 AND dec >= 24.12",
    "south_east": "ra >= 56.75 AND dec < 24.12",
    "north_east_inner": (
        "ra >= 56.75 AND ra < 58.25 AND dec >= 24.12"
    ),
    "north_east_outer": "ra >= 58.25 AND dec >= 24.12",
}

expected_chunk_counts = {
    "south_west": 50_280,
    "north_west": 61_960,
    "south_east": 65_642,
    "north_east_inner": 42_077,
    "north_east_outer": 35_594,
}

print(
    "Expected total: "
    f"{sum(expected_chunk_counts.values()):,}"
)

## Load cached chunks or query missing data

Each validated archive result is stored separately in `data/raw/`. On a
rerun, an existing file is loaded rather than overwritten or queried
again. If a file is missing, an asynchronous ARI job retrieves exactly that
chunk in VOTable table-data format, which avoids a Windows binary-integer
parsing issue encountered for Gaia's 64-bit `source_id`.

In [ ]:
def load_or_download_chunk(
    chunk_name,
    chunk_condition,
    expected_rows,
):
    # Load a validated raw chunk, or download it if absent.

    output_path = (
        raw_data_dir
        / f"gaia_dr3_pleiades_3deg_{chunk_name}.fits"
    )

    if output_path.exists():
        table = Table.read(output_path, format="fits")
        source = "local FITS file"
    else:
        query = build_chunk_query(chunk_condition)
        job = ari_gaia.submit_job(
            query,
            FORMAT="votable/td",
        )
        job.run()
        job.wait(timeout=1800)
        job.raise_if_error()
        table = job.fetch_result().to_table()
        source = f"ARI job {job.job_id}"

        if len(table) != expected_rows:
            raise ValueError(
                f"{chunk_name}: expected {expected_rows} rows, "
                f"received {len(table)}."
            )
        if table.colnames != gaia_columns:
            raise ValueError(
                f"{chunk_name}: unexpected column structure."
            )

        table.write(
            output_path,
            format="fits",
            overwrite=False,
        )

    if len(table) != expected_rows:
        raise ValueError(
            f"{chunk_name}: cached row count is invalid."
        )
    if table.colnames != gaia_columns:
        raise ValueError(
            f"{chunk_name}: cached columns are invalid."
        )

    print(
        f"{chunk_name}: {len(table):,} rows "
        f"from {source}"
    )
    return table

In [ ]:
chunk_tables = {}

for chunk_name, chunk_condition in chunk_conditions.items():
    chunk_tables[chunk_name] = load_or_download_chunk(
        chunk_name,
        chunk_condition,
        expected_chunk_counts[chunk_name],
    )

## Combine and validate the northeast region

The two northeast transfer chunks are vertically stacked into one
convenience file. Their original raw files are retained as provenance.

In [ ]:
north_east_table = vstack(
    [
        chunk_tables["north_east_inner"],
        chunk_tables["north_east_outer"],
    ],
    join_type="exact",
    metadata_conflicts="silent",
)

expected_north_east_rows = (
    expected_chunk_counts["north_east_inner"]
    + expected_chunk_counts["north_east_outer"]
)
north_east_duplicate_count = (
    len(north_east_table)
    - np.unique(north_east_table["source_id"]).size
)

if (
    len(north_east_table) != expected_north_east_rows
    or north_east_duplicate_count != 0
):
    raise ValueError("Northeast combination failed validation.")

north_east_path = (
    raw_data_dir
    / "gaia_dr3_pleiades_3deg_north_east.fits"
)

if not north_east_path.exists():
    north_east_table.write(
        north_east_path,
        format="fits",
        overwrite=False,
    )

north_east_check = Table.read(
    north_east_path,
    format="fits",
)

print(f"Northeast rows: {len(north_east_check):,}")
print(
    "Duplicate northeast source IDs: "
    f"{north_east_duplicate_count}"
)

## Combine the complete raw sample

All five non-overlapping transfer chunks are stacked into a single raw
table. The result must contain 255,553 rows and no duplicated Gaia source
identifiers before it is saved.

In [ ]:
full_raw_table = vstack(
    list(chunk_tables.values()),
    join_type="exact",
    metadata_conflicts="silent",
)

expected_total = sum(expected_chunk_counts.values())
duplicate_source_count = (
    len(full_raw_table)
    - np.unique(full_raw_table["source_id"]).size
)

if (
    len(full_raw_table) != expected_total
    or duplicate_source_count != 0
):
    raise ValueError("Full raw table failed validation.")

full_raw_path = (
    raw_data_dir
    / "gaia_dr3_pleiades_3deg_all_sources.fits"
)

if not full_raw_path.exists():
    full_raw_table.write(
        full_raw_path,
        format="fits",
        overwrite=False,
    )

saved_full_table = Table.read(
    full_raw_path,
    format="fits",
)
saved_ids_match = np.array_equal(
    np.sort(saved_full_table["source_id"]),
    np.sort(full_raw_table["source_id"]),
)

if not saved_ids_match:
    raise ValueError("Saved full table does not match chunks.")

full_raw_table = saved_full_table

print(f"Combined rows: {len(full_raw_table):,}")
print(f"Duplicate source IDs: {duplicate_source_count}")
print(f"Complete raw file: {full_raw_path}")

## Initial parallax distribution

A parallax histogram provides a first view of the distance structure. The
displayed range of -5 to 15 mas excludes only extreme values from the
visualization; it is not a membership or data-quality cut. A logarithmic
vertical axis reveals less-populated structure beside the field-star peak.

### Initial interpretation

- The broad distribution near 0-1 mas is dominated by field stars.
- A narrower peak near 7.3-7.5 mas is consistent with a population at the
  shared distance of the Pleiades, but is not a membership determination.
- Negative measured parallaxes arise from observational uncertainty and do
  not represent physical negative distances.
- Parallax must later be considered with proper motion, measurement quality,
  clustering, and the colour-magnitude diagram.

In [ ]:
parallax_values = np.ma.asarray(
    full_raw_table["parallax"]
).filled(np.nan)
finite_parallax = parallax_values[
    np.isfinite(parallax_values)
]

parallax_plot_min = -5.0
parallax_plot_max = 15.0
parallax_for_plot = finite_parallax[
    (finite_parallax >= parallax_plot_min)
    & (finite_parallax <= parallax_plot_max)
]

print(f"Total sources: {len(full_raw_table):,}")
print(f"Finite parallaxes: {len(finite_parallax):,}")
print(
    "Fraction of finite parallaxes displayed: "
    f"{len(parallax_for_plot) / len(finite_parallax):.3%}"
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(
    parallax_for_plot,
    bins=200,
    color="steelblue",
    alpha=0.8,
)
ax.set_yscale("log")
ax.set_xlabel("Parallax (mas)")
ax.set_ylabel("Number of sources")
ax.set_title(
    "Gaia DR3 parallax distribution around the Pleiades"
)
ax.grid(alpha=0.2)
fig.tight_layout()

parallax_figure_path = (
    figures_dir / "01_parallax_distribution.png"
)
fig.savefig(
    parallax_figure_path,
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## Initial proper-motion distribution

Gaia's `pmra` is the cosine-corrected component
$mu_{alpha*} = mu_alphacos(delta)$. A hexagonal density plot is used
because an ordinary scatter plot would strongly overplot more than 220,000
sources. Display limits remove only extreme values from the visualization.

### Initial interpretation

- The broad concentration near small proper motions is dominated by field
  stars at many distances and with varied physical velocities.
- A compact overdensity near
  $(mu_{alpha*},mu_delta)\approx(20,-45)$ mas yr$^{-1}$ is consistent
  with the expected Pleiades population.
- The proper-motion clump and parallax peak provide complementary evidence,
  but neither visualization alone defines confirmed members.

In [ ]:
pmra_values = np.ma.asarray(
    full_raw_table["pmra"]
).filled(np.nan)
pmdec_values = np.ma.asarray(
    full_raw_table["pmdec"]
).filled(np.nan)

finite_pm_mask = (
    np.isfinite(pmra_values)
    & np.isfinite(pmdec_values)
)
finite_pmra = pmra_values[finite_pm_mask]
finite_pmdec = pmdec_values[finite_pm_mask]

pm_display_mask = (
    (finite_pmra >= -30.0)
    & (finite_pmra <= 60.0)
    & (finite_pmdec >= -70.0)
    & (finite_pmdec <= 30.0)
)
pmra_for_plot = finite_pmra[pm_display_mask]
pmdec_for_plot = finite_pmdec[pm_display_mask]

print(
    "Sources with both proper motions: "
    f"{finite_pm_mask.sum():,}"
)
print(
    "Fraction of finite proper motions displayed: "
    f"{len(pmra_for_plot) / len(finite_pmra):.3%}"
)

fig, ax = plt.subplots(figsize=(8, 6))
density = ax.hexbin(
    pmra_for_plot,
    pmdec_for_plot,
    gridsize=160,
    bins="log",
    mincnt=1,
    cmap="viridis",
)
ax.set_xlabel(r"$\mu_{\alpha*}$ (mas yr$^{-1}$)")
ax.set_ylabel(r"$\mu_\delta$ (mas yr$^{-1}$)")
ax.set_title(
    "Gaia DR3 proper-motion distribution around the Pleiades"
)
ax.grid(alpha=0.15)

colour_bar = fig.colorbar(density, ax=ax)
colour_bar.set_label("Sources per hexagonal bin")
fig.tight_layout()

proper_motion_figure_path = (
    figures_dir / "02_proper_motion_distribution.png"
)
fig.savefig(
    proper_motion_figure_path,
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## Notebook outcome

The validated raw Gaia DR3 sample contains 255,553 unique sources within
3 degrees of the adopted Pleiades centre. The raw astrometric, uncertainty,
quality, and photometric columns are stored in
`data/raw/gaia_dr3_pleiades_3deg_all_sources.fits`.

Initial plots reveal a narrow parallax feature near 7.4 mas and a compact
proper-motion overdensity near `(20, -45)` mas/yr. These are candidate
cluster signatures, not final membership criteria. Missing values and
quality filtering will be handled explicitly in Notebook 02.

In [ ]:
print(f"Raw catalogue: {full_raw_path}")
print(f"Rows: {len(full_raw_table):,}")
print(f"Columns: {len(full_raw_table.colnames)}")
print(f"Parallax figure: {parallax_figure_path}")
print(f"Proper-motion figure: {proper_motion_figure_path}")